# NB07 — Profiling and Performance Optimization

**Module 16 — Research Software Engineering**

---

## Motivation

Premature optimization is the root of all evil — but so is not knowing where your bottlenecks are. A bioinformatics pipeline that processes 100 genomes should not take a week to run because of an O(N²) loop that could be O(N). Profiling tells you *where* time is actually spent, so you optimize the right function, not the wrong one.

## Intuition

Profiling is measurement. You run your code under a profiler, which records how many times each function is called and how long it takes. You then look at the hotspots — the 20% of functions that account for 80% of runtime — and only those get optimized. The rule: **never optimize without measuring first**.

## Biological Background

K-mer counting is a fundamental operation in bioinformatics: used in genome assembly (de Bruijn graphs), species identification (MinHash/sourmash), error correction, and taxonomic classification (Kraken2). Jellyfish can count k-mers in a 3 Gb genome in seconds. Understanding why the naive approach is slow — and why dictionary-based counting is fast — is the foundation for appreciating these tools.

## Mathematical / Computational Explanation

- **Naive k-mer counting:** for each position i in the sequence (N positions), extract a substring of length k (O(k) per extraction), then search a list for it (O(M) where M = number of distinct k-mers seen). Total: O(N × k × M) in the worst case.
- **Dictionary-based counting:** for each position i, extract a substring of length k (O(k) per extraction with slicing, O(1) amortized with rolling hash), look it up in a hash table (O(1) average). Total: O(N × k) for simple slicing, O(N) with a rolling hash.

---

In [ ]:
import time
import timeit
import random
import numpy as np
from collections import defaultdict

rng = np.random.default_rng(42)

def random_dna(length: int, rng=rng) -> str:
    """Generate a random DNA sequence of the given length."""
    return ''.join(rng.choice(list('ATCG'), size=length))


# Method 1: Naive k-mer counting using a list (O(N*k*M))
def kmer_count_naive(seq: str, k: int) -> dict[str, int]:
    """Count k-mers using a list (demonstrably slow for large sequences)."""
    kmers_seen: list[str] = []
    counts: list[int] = []
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i + k]
        if kmer in kmers_seen:
            idx = kmers_seen.index(kmer)  # O(M) search!
            counts[idx] += 1
        else:
            kmers_seen.append(kmer)
            counts.append(1)
    return dict(zip(kmers_seen, counts))


# Method 2: Dictionary-based counting (O(N*k) with hash table)
def kmer_count_dict(seq: str, k: int) -> dict[str, int]:
    """Count k-mers using a Python dict (O(N*k) on average)."""
    counts: dict[str, int] = {}
    for i in range(len(seq) - k + 1):
        kmer = seq[i:i + k]
        counts[kmer] = counts.get(kmer, 0) + 1
    return counts


# Verify correctness: both methods should give identical results
test_seq = random_dna(100)
result_naive = kmer_count_naive(test_seq, k=4)
result_dict  = kmer_count_dict(test_seq, k=4)
assert result_naive == result_dict, 'Methods disagree!'
print(f'Correctness check passed. Distinct 4-mers in 100-bp sequence: {len(result_dict)}')

In [ ]:
# Benchmark both methods over a range of sequence lengths
# Using timeit for reliable timing

lengths = [100, 300, 1000, 3000, 10000]
k = 7
n_repeats = 3

times_naive = []
times_dict  = []

print(f'Benchmarking k-mer counting (k={k}), {n_repeats} repeats each:\n')
print(f'{"Length":>8}  {"Naive (ms)":>12}  {"Dict (ms)":>12}  {"Speedup":>8}')
print('-' * 50)

for L in lengths:
    seq = random_dna(L)

    # Naive: only for small sequences (would take too long otherwise)
    if L <= 3000:
        t_naive = timeit.timeit(lambda: kmer_count_naive(seq, k), number=n_repeats)
        t_naive_ms = t_naive / n_repeats * 1000
    else:
        t_naive_ms = float('nan')  # too slow to benchmark

    t_dict = timeit.timeit(lambda: kmer_count_dict(seq, k), number=n_repeats)
    t_dict_ms = t_dict / n_repeats * 1000

    times_naive.append(t_naive_ms)
    times_dict.append(t_dict_ms)

    speedup = t_naive_ms / t_dict_ms if not (t_naive_ms != t_naive_ms) else float('nan')
    naive_str = f'{t_naive_ms:.3f}' if t_naive_ms == t_naive_ms else 'N/A (too slow)'
    print(f'{L:>8}  {naive_str:>12}  {t_dict_ms:>12.3f}  {speedup:>8.1f}x')

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Profiling and Optimization: k-mer Counting', fontsize=13, fontweight='bold')

# Filter out NaN for naive
valid_mask = [not (v != v) for v in times_naive]
valid_lengths = [l for l, m in zip(lengths, valid_mask) if m]
valid_naive   = [t for t, m in zip(times_naive, valid_mask) if m]
valid_dict_for_naive = [t for t, m in zip(times_dict, valid_mask) if m]

# --- Panel 1: Execution time vs sequence length (log-log) ---
ax1 = axes[0]
ax1.loglog(valid_lengths, valid_naive, 'o-', color='#e74c3c', label='Naive (list)', lw=2, ms=6)
ax1.loglog(lengths, times_dict, 's-', color='#27ae60', label='Dict-based', lw=2, ms=6)
ax1.set_xlabel('Sequence length (bp)', fontsize=10)
ax1.set_ylabel('Time (ms)', fontsize=10)
ax1.set_title('Execution Time vs Length\n(log-log scale)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')

# Add Big-O annotations
ax1.text(200, max(valid_naive)*0.5, 'O(N²)', color='#e74c3c', fontsize=11, fontweight='bold')
ax1.text(200, times_dict[-1]*5, 'O(N)', color='#27ae60', fontsize=11, fontweight='bold')

# --- Panel 2: Memory usage estimate ---
ax2 = axes[1]
# Naive stores lists; dict stores hash table
# Estimate: 4^k possible k-mers, each string is ~k bytes + overhead
k_values = [4, 5, 6, 7, 8, 9, 10]
distinct_kmers = [min(4**k, 10000) for k in k_values]  # bounded by seq length
memory_bytes_est = [dk * (k + 28 + 28) for dk, k in zip(distinct_kmers, k_values)]  # key + val

ax2.semilogy(k_values, memory_bytes_est, 'D-', color='#3498db', lw=2, ms=6)
ax2.set_xlabel('k-mer size k', fontsize=10)
ax2.set_ylabel('Estimated memory (bytes, log)', fontsize=10)
ax2.set_title('Memory vs k-mer Size', fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

# --- Panel 3: Speedup ratio ---
ax3 = axes[2]
valid_speedup = [n / d for n, d in zip(valid_naive, valid_dict_for_naive)]
ax3.bar(range(len(valid_lengths)), valid_speedup, color='#9b59b6', alpha=0.8)
ax3.set_xticks(range(len(valid_lengths)))
ax3.set_xticklabels([str(l) for l in valid_lengths])
ax3.set_xlabel('Sequence length (bp)', fontsize=10)
ax3.set_ylabel('Speedup (naive / dict)', fontsize=10)
ax3.set_title('Speedup: Dict vs Naive', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for i, sp in enumerate(valid_speedup):
    ax3.text(i, sp + 0.5, f'{sp:.1f}x', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('profiling_optimization.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E07.

1. Profile a naive GC-content-over-sliding-window implementation using `timeit`. Identify the bottleneck and optimize it.
2. What is the Big-O complexity of `kmer_count_dict` for k=21? How many distinct k-mers are possible for a 1 Mbp genome with k=21?
3. When would you use `cProfile` vs `timeit`? What does each tell you?

## Quiz

1. What is the rule about optimization and measurement?
2. What is the complexity of Python `list.index()`?
3. What is the average-case complexity of Python `dict` lookup?
4. What does `timeit.timeit(fn, number=N)` return?
5. What does a log-log plot reveal about the growth rate of a function?

## Reflection

*After completing:* The speedup from dict-based counting grows with sequence length. Explain in plain English *why* this is the case — reference the Big-O analysis.

## References

- Python timeit module: https://docs.python.org/3/library/timeit.html
- Marcais & Kingsford (2011) "A fast, lock-free approach for efficient parallel counting of occurrences of k-mers." Jellyfish paper.